# SFT on Agent Traces (TRL + LoRA) — Colab T4

Minimal lab for the Hugging Face **Training Agents** SFT-on-traces recipe:

1. Load example agent traces (JSONL)
2. Convert → **prompt-completion** pairs (one row per assistant turn)
3. **LoRA / QLoRA SFT** on `Qwen2.5-1.5B-Instruct` with **completion-only loss**
4. Score **format correctness** (tool syntax, structure)

**Runtime → Change runtime type → GPU (T4).**  
Demo train is ~30 steps so you finish in minutes, not hours.

## 0. Install dependencies

In [1]:
# Quiet install; restart runtime only if Colab complains about binary mismatches.
%pip install -q -U transformers datasets trl peft accelerate bitsandbytes sentencepiece

import torch
print("torch:", torch.__version__)
print("cuda :", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"vram : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.0 MB/s eta 0:00:00
torch: 2.11.0+cu128
cuda : True Tesla T4
vram : 15.5 GB free / 15.6 GB total


## 1. Project layout + sample agent traces

Each line is one **trace**: a multi-turn coding-agent session with optional tool turns.  
Tool calls use a simple readable format (easy to eval without JSON parsing):

```text
invoke tool list_dir with path is /tmp
```

In [2]:
from pathlib import Path
import json

ROOT = Path("sft-agent-traces")
(ROOT / "data").mkdir(parents=True, exist_ok=True)
(ROOT / "outputs").mkdir(parents=True, exist_ok=True)
(ROOT / "src").mkdir(parents=True, exist_ok=True)

TRACES = [
  {
    "trace_id": "trace-001",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "List the files in /tmp and tell me how many there are."},
      {"role": "assistant", "content": "I'll list the contents of /tmp for you.\n\ninvoke tool list_dir with path is /tmp"},
      {"role": "tool", "content": "file_a.txt\nfile_b.py\nnotes.md"},
      {"role": "assistant", "content": "There are **3** files in `/tmp`:\n1. `file_a.txt`\n2. `file_b.py`\n3. `notes.md`"},
    ],
  },
  {
    "trace_id": "trace-002",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "What does README.md say in its first line?"},
      {"role": "assistant", "content": "I'll read the README.\n\ninvoke tool read_file with path is README.md"},
      {"role": "tool", "content": "# Agent SFT Demo\n\nMinimal reproducible project."},
      {"role": "assistant", "content": "The first line of `README.md` is:\n\n```\n# Agent SFT Demo\n```"},
    ],
  },
  {
    "trace_id": "trace-003",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "Find where SFTConfig is used and quote the learning rate."},
      {"role": "assistant", "content": "Searching for SFTConfig.\n\ninvoke tool grep with pattern is SFTConfig"},
      {"role": "tool", "content": "src/train_sft.py:42:    args = SFTConfig(\nsrc/train_sft.py:48:        learning_rate=2e-4,"},
      {"role": "assistant", "content": "Found it. Reading that section.\n\ninvoke tool read_file with path is src/train_sft.py"},
      {"role": "tool", "content": "learning_rate=2e-4,"},
      {"role": "assistant", "content": "The learning rate is set to **`2e-4`** in `SFTConfig`."},
    ],
  },
  {
    "trace_id": "trace-004",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "Add is_even(n) in utils_math.py."},
      {"role": "assistant", "content": "Creating the helper.\n\ninvoke tool write_file with path is utils_math.py with content is def is_even(n: int) -> bool:\\n    return n % 2 == 0\\n"},
      {"role": "tool", "content": "Wrote 3 lines to utils_math.py"},
      {"role": "assistant", "content": "Done. `utils_math.py` now defines `is_even`."},
    ],
  },
  {
    "trace_id": "trace-005",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "Run the unit tests and summarize."},
      {"role": "assistant", "content": "Running tests.\n\ninvoke tool run_shell with command is pytest -q"},
      {"role": "tool", "content": "5 passed in 0.42s"},
      {"role": "assistant", "content": "All tests passed: **5 passed** in 0.42s."},
    ],
  },
  {
    "trace_id": "trace-006",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "In one sentence, what is supervised fine-tuning (SFT)?"},
      {"role": "assistant", "content": "Supervised fine-tuning (SFT) is training a pretrained language model on labeled input–output pairs so it learns to produce the desired completions for those prompts."},
    ],
  },
  {
    "trace_id": "trace-007",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "Why do we mask the prompt when doing SFT on agent traces?"},
      {"role": "assistant", "content": "We mask the prompt (labels=-100) so loss is computed only on assistant tokens, teaching the model how to respond as the agent rather than to regenerate user/system/tool text."},
    ],
  },
  {
    "trace_id": "trace-008",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "Create hello.txt with 'hello world' and verify."},
      {"role": "assistant", "content": "Writing then reading back.\n\ninvoke tool write_file with path is hello.txt with content is hello world\\n"},
      {"role": "tool", "content": "Wrote 1 lines to hello.txt"},
      {"role": "assistant", "content": "invoke tool read_file with path is hello.txt"},
      {"role": "tool", "content": "hello world"},
      {"role": "assistant", "content": "Verified: `hello.txt` contains `hello world`."},
    ],
  },
  {
    "trace_id": "trace-009",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "Summarize git status in one sentence."},
      {"role": "assistant", "content": "Checking status.\n\ninvoke tool run_shell with command is git status --short"},
      {"role": "tool", "content": " M src/train_sft.py\n?? data/new_traces.jsonl"},
      {"role": "assistant", "content": "One modified tracked file and one untracked file."},
    ],
  },
  {
    "trace_id": "trace-010",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "How many .py files are under src/?"},
      {"role": "assistant", "content": "Counting Python files.\n\ninvoke tool run_shell with command is find src -name '*.py' | wc -l"},
      {"role": "tool", "content": "4"},
      {"role": "assistant", "content": "There are **4** `.py` files under `src/`."},
    ],
  },
  {
    "trace_id": "trace-011",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "Write load_jsonl(path) that yields parsed objects."},
      {"role": "assistant", "content": "```python\nimport json\nfrom pathlib import Path\nfrom typing import Any, Iterator\n\ndef load_jsonl(path: str | Path) -> Iterator[dict[str, Any]]:\n    with open(path, encoding='utf-8') as f:\n        for line in f:\n            line = line.strip()\n            if line:\n                yield json.loads(line)\n```"},
    ],
  },
  {
    "trace_id": "trace-012",
    "source": "synthetic-coding-agent",
    "messages": [
      {"role": "system", "content": "You are a helpful coding agent. Use tools when needed. Respond with tool calls in this format:\ninvoke tool tool_name with arg1 is value1\nWhen finished, give a clear final answer."},
      {"role": "user", "content": "The script fails with NameError: x is not defined. Fix broken.py."},
      {"role": "assistant", "content": "Inspecting the file.\n\ninvoke tool read_file with path is broken.py"},
      {"role": "tool", "content": "def main():\n    print(x + 1)"},
      {"role": "assistant", "content": "`x` is undefined. Fixing it.\n\ninvoke tool write_file with path is broken.py with content is def main(x: int = 0) -> None:\\n    print(x + 1)\\n"},
      {"role": "tool", "content": "Wrote file"},
      {"role": "assistant", "content": "Fixed by giving `main` a default `x: int = 0`."},
    ],
  },
]

traces_path = ROOT / "data" / "example_traces.jsonl"
with traces_path.open("w", encoding="utf-8") as f:
    for row in TRACES:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Wrote {len(TRACES)} traces → {traces_path}")
print("Sample roles:", [m["role"] for m in TRACES[0]["messages"]])

Wrote 12 traces → sft-agent-traces/data/example_traces.jsonl
Sample roles: ['system', 'user', 'assistant', 'tool', 'assistant']


## 2. Convert traces → prompt-completion

For **each assistant message** at index `i`:

- `prompt` = messages before `i` (system / user / prior assistant / tool)
- `completion` = that assistant message only

TRL will then apply **completion-only loss** (mask prompt tokens with `labels=-100`).

In [3]:
from typing import Any

def expand_trace_to_prompt_completion(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """One supervised row per assistant turn."""
    examples = []
    norm = []
    for m in messages:
        content = m.get("content") or ""
        if not isinstance(content, str):
            content = json.dumps(content)
        norm.append({"role": m["role"], "content": content})

    for i, msg in enumerate(norm):
        if msg["role"] != "assistant" or not msg["content"].strip():
            continue
        prompt = norm[:i]
        if not any(m["role"] in ("user", "system") for m in prompt):
            continue
        examples.append({"prompt": prompt, "completion": [msg]})
    return examples


raw = [json.loads(l) for l in traces_path.read_text().splitlines() if l.strip()]
examples = []
for t in raw:
    examples.extend(expand_trace_to_prompt_completion(t["messages"]))

pc_path = ROOT / "data" / "prompt_completion.jsonl"
with pc_path.open("w", encoding="utf-8") as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"{len(raw)} traces → {len(examples)} prompt-completion rows")
print("First example prompt roles:", [m["role"] for m in examples[0]["prompt"]])
print("First completion preview:", examples[0]["completion"][0]["content"][:120].replace("\n", " "))

12 traces → 24 prompt-completion rows
First example prompt roles: ['system', 'user']
First completion preview: I'll list the contents of /tmp for you.  invoke tool list_dir with path is /tmp


### (Optional) Peek at completion-only masking

No training here — just tokenize one example and show which side of the boundary is supervised.

In [4]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # switch to google/gemma-2-2b-it if you accept the license
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

ex = examples[0]
prompt_text = tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
full_text = tok.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=False, add_generation_prompt=False)

prompt_ids = tok(prompt_text, add_special_tokens=False)["input_ids"]
full_ids = tok(full_text, add_special_tokens=False)["input_ids"]
n_prompt = len(prompt_ids)

print(f"prompt tokens (MASK / labels=-100): {n_prompt}")
print(f"completion tokens (LOSS):           {max(len(full_ids) - n_prompt, 0)}")
print("--- prompt tail ---")
print(prompt_text[-200:])
print("--- completion ---")
print(ex["completion"][0]["content"][:300])

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

prompt tokens (MASK / labels=-100): 67
completion tokens (LOSS):           22
--- prompt tail ---
:
invoke tool tool_name with arg1 is value1
When finished, give a clear final answer.<|im_end|>
<|im_start|>user
List the files in /tmp and tell me how many there are.<|im_end|>
<|im_start|>assistant

--- completion ---
I'll list the contents of /tmp for you.

invoke tool list_dir with path is /tmp


## 3. LoRA SFT with TRL

Key flag: `completion_only_loss=True` — train only on assistant/completion tokens.

T4 recipe: 4-bit QLoRA + LoRA r=16 + batch 1 + grad accum 8 + `max_length=1024` + gradient checkpointing.

In [10]:
import json
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = str(ROOT / "outputs" / "lora-sft")
MAX_STEPS = 30          # raise for a real run
MAX_LENGTH = 1024

rows = [json.loads(l) for l in pc_path.read_text().splitlines() if l.strip()]
dataset = Dataset.from_list(rows)
split = dataset.train_test_split(test_size=0.15, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(f"train={len(train_ds)} eval={len(eval_ds)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4-friendly compute dtype
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

# fp16=False avoids GradScaler + bf16 crash on QLoRA/T4:
# NotImplementedError: _amp_foreach_non_finite_check_and_unscale_cuda
# not implemented for 'BFloat16'
sft_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=max(int(MAX_STEPS * 0.05), 1),
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=max(MAX_STEPS // 2, 1),
    save_steps=MAX_STEPS,
    save_total_limit=1,
    max_length=MAX_LENGTH,
    completion_only_loss=True,   # mask prompt; train on completion only
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=42,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

result = trainer.train()
print(result.metrics)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

run_card = {
    "base_model": MODEL_ID,
    "max_steps": MAX_STEPS,
    "completion_only_loss": True,
    "metrics": result.metrics,
}
Path(OUTPUT_DIR, "run_card.json").write_text(json.dumps(run_card, indent=2))
print("Saved adapters →", OUTPUT_DIR)


train=20 eval=4


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
15,0.088770,0.775737,0.479038,12185.000000,0.829038
30,0.009633,1.055096,0.302480,24370.000000,0.800577


{'train_runtime': 139.9188, 'train_samples_per_second': 1.715, 'train_steps_per_second': 0.214, 'total_flos': 194297922078720.0, 'train_loss': 0.4973199222081651, 'epoch': 10.0}
Saved adapters → sft-agent-traces/outputs/lora-sft


## 4. Format-correctness evaluation

Cheap structural checks (no LLM judge). Good first signal after SFT on agent traces.

In [11]:
import re
from peft import PeftModel

TOOL_INVOKE_RE = re.compile(
    r"invoke\s+tool\s+(?P<name>[A-Za-z_][A-Za-z0-9_]*)\s+with\s+.+(?:\bis\b)\s+.+",
    re.IGNORECASE | re.DOTALL,
)
TOOL_NAME_RE = re.compile(r"invoke\s+tool\s+(?P<name>[A-Za-z_][A-Za-z0-9_]*)", re.I)
ROLE_LEAK_RE = re.compile(r"(<\|im_start\|>|<\|im_end\|>|^\s*user\s*:|^\s*system\s*:)", re.I | re.M)

SYSTEM = (
    "You are a helpful coding agent. Use tools when needed. "
    "Respond with tool calls in this format:\n"
    "invoke tool tool_name with arg1 is value1\n"
    "When finished, give a clear final answer."
)

EVAL_PROMPTS = [
    {"id": "list-dir", "user": "List the files under ./data and count them.", "expect_tool": True},
    {"id": "read-file", "user": "Read requirements.txt and summarize dependencies.", "expect_tool": True},
    {"id": "define-sft", "user": "In two sentences, explain completion-only loss in SFT.", "expect_tool": False},
    {"id": "write-helper", "user": "Create double.py with a function that doubles an int.", "expect_tool": True},
    {"id": "no-tool", "user": "What is LoRA in one short paragraph? Do not use tools.", "expect_tool": False},
    {"id": "grep", "user": "Search the repo for completion_only_loss.", "expect_tool": True},
]

def score_generation(text: str) -> dict:
    text = (text or "").strip()
    has_tool = bool(TOOL_INVOKE_RE.search(text))
    tool_name_only = bool(TOOL_NAME_RE.search(text))
    without_tool = "\n".join(ln for ln in text.splitlines() if "invoke tool" not in ln.lower()).strip()
    has_answer = len(without_tool) >= 20
    checks = {
        "non_empty": len(text) > 0,
        "no_user_role_leak": not bool(ROLE_LEAK_RE.search(text)),
        "tool_or_answer": has_tool or has_answer,
        "tool_syntax_ok": (not tool_name_only) or has_tool,
        "final_answer_ok": has_tool or has_answer,
        "detected_tool": has_tool,
    }
    checks["pass"] = all(checks[k] for k in [
        "non_empty", "no_user_role_leak", "tool_or_answer", "tool_syntax_ok", "final_answer_ok"
    ])
    checks["preview"] = text[:180].replace("\n", "\\n")
    return checks

@torch.inference_mode()
def generate(user_text: str, max_new_tokens: int = 256) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = out[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

# `model` is already the PeftModel from SFTTrainer; reuse it for eval.
model.eval()

results = []
for item in EVAL_PROMPTS:
    text = generate(item["user"])
    s = score_generation(text)
    s["id"] = item["id"]
    s["generation"] = text
    s["tool_when_expected"] = (not item["expect_tool"]) or s["detected_tool"]
    results.append(s)
    print(f"[{'PASS' if s['pass'] else 'FAIL'}] {item['id']}: {s['preview'][:100]}")

keys = ["non_empty", "no_user_role_leak", "tool_or_answer", "tool_syntax_ok", "final_answer_ok", "pass", "tool_when_expected"]
summary = {k: sum(1 for r in results if r.get(k)) / len(results) for k in keys}
print("\nSummary:")
for k, v in summary.items():
    print(f"  {k:22s} {v:5.1%}")

eval_path = ROOT / "outputs" / "format_eval.json"
eval_path.write_text(json.dumps({"summary": summary, "per_prompt": results}, indent=2, ensure_ascii=False))
print("Wrote", eval_path)

[PASS] list-dir: I'll list the contents of ./data for you.\n\ninvoke tool list_dir with path is ./data
[PASS] read-file: I'll read the file.\n\ninvoke tool read_file with path is requirements.txt
[PASS] define-sft: Completion-only loss in sequence-to-sequence (SFT) models focuses on minimizing the loss for the tok
[PASS] write-helper: ```python\ndef double(x):\n    return x * 2\n```
[PASS] no-tool: LoRA stands for "Low-Rank Adaptation," where pretrained weights are adapted to the specific architec
[PASS] grep: Searching for `completion_only_loss`.\n\ninvoke tool git_search with path is repos/main/src/

Summary:
  non_empty              100.0%
  no_user_role_leak      100.0%
  tool_or_answer         100.0%
  tool_syntax_ok         100.0%
  final_answer_ok        100.0%
  pass                   100.0%
  tool_when_expected     83.3%
Wrote sft-agent-traces/outputs/format_eval.json


## 5. What to try next

| Idea | How |
|------|-----|
| More steps | Set `MAX_STEPS = 100–300` |
| Gemma-2-2B | `MODEL_ID = "google/gemma-2-2b-it"` after `huggingface-cli login` |
| Your traces | Replace `TRACES` / load a Hub dataset, keep the same converter |
| Real task eval | Run the agent in a sandbox and score tool success, not just format |
| Next post-training rung | Preference (DPO) or RL (GRPO) on the same prompt distribution |

### Mental model

```
traces  →  prompt/completion  →  completion-only CE loss  →  LoRA adapters  →  format checks
```

SFT teaches **distribution / format**. Tool *correctness* and multi-step reliability usually need better data, environment feedback, or RL — but this baseline is the first rung.

In [12]:
# ===== Chat with fine-tuned agent (simulated tools) =====
# Prerequisites: training finished (adapter at ROOT/outputs/lora-sft).
# Reuses model/tokenizer in memory if present; otherwise reloads.

import re
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = str(ROOT / "outputs" / "lora-sft")
MAX_NEW_TOKENS = 256
MAX_TOOL_ROUNDS = 4

SYSTEM = (
    "You are a helpful coding agent. Use tools when needed. "
    "Respond with tool calls in this format:\n"
    "invoke tool tool_name with arg1 is value1\n"
    "When finished, give a clear final answer."
)

TOOL_RE = re.compile(
    r"invoke\s+tool\s+(?P<name>[A-Za-z_][A-Za-z0-9_]*)\s+with\s+(?P<body>.+)",
    re.IGNORECASE | re.DOTALL,
)

def parse_tool_call(text: str):
    m = TOOL_RE.search(text or "")
    if not m:
        return None
    name = m.group("name")
    body = m.group("body").strip()
    parts = re.split(r"\s+with\s+", body, flags=re.IGNORECASE)
    args = {}
    for part in parts:
        if " is " in part:
            k, v = part.split(" is ", 1)
            args[k.strip()] = v.strip()
        elif part.strip():
            args["arg"] = part.strip()
    return {"name": name, "arguments": args}

def simulate_tool(name: str, arguments: dict) -> str:
    """Fake tools so multi-turn works without a real sandbox."""
    n = name.lower()
    path = arguments.get("path", arguments.get("arg", "."))
    if n in ("list_dir", "ls", "list"):
        return "example_traces.jsonl\nprompt_completion.jsonl\nnotes.md"
    if n in ("read_file", "read", "cat"):
        if "requirements" in path:
            return "torch\ntransformers\ntrl\npeft\nbitsandbytes\n"
        if "readme" in path.lower():
            return "# Agent SFT Demo\n\nMinimal SFT-on-traces project.\n"
        return f"[simulated contents of {path}]\nline 1\nline 2\n"
    if n in ("write_file", "write", "create_file"):
        content = arguments.get("content", "")
        return f"Wrote {max(len(content.splitlines()), 1)} lines to {path}"
    if n in ("run_shell", "shell", "bash", "run"):
        cmd = arguments.get("command", arguments.get("arg", ""))
        if "pytest" in cmd:
            return "5 passed in 0.42s"
        if "git status" in cmd:
            return " M src/train_sft.py\n?? data/new_traces.jsonl"
        return f"[simulated stdout for: {cmd}]"
    if n in ("grep", "search", "git_search"):
        return "src/train_sft.py:48:        completion_only_loss=True,\n"
    return f"[simulated] tool={name!r} args={arguments!r} → ok"

# --- load model if needed ---
need_load = "model" not in dir() or "tokenizer" not in dir()
if need_load:
    print("Loading base + adapter from disk...")
    tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE,
        quantization_config=bnb,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    if Path(ADAPTER).exists():
        model = PeftModel.from_pretrained(model, ADAPTER)
        print("Loaded adapter:", ADAPTER)
    else:
        print("WARNING: adapter not found — base model only:", ADAPTER)
else:
    print("Reusing model/tokenizer already in memory.")

model.eval()

@torch.inference_mode()
def generate_reply(messages, max_new_tokens=MAX_NEW_TOKENS) -> str:
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = out[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def chat_once(user_text: str, max_tool_rounds: int = MAX_TOOL_ROUNDS) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_text},
    ]
    final = ""
    for r in range(max_tool_rounds + 1):
        reply = generate_reply(messages)
        final = reply
        print(f"\n--- assistant (round {r+1}) ---\n{reply}")
        tool = parse_tool_call(reply)
        if tool is None:
            break
        result = simulate_tool(tool["name"], tool["arguments"])
        print(f"\n--- tool: {tool['name']} {tool['arguments']} ---\n{result}")
        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "tool", "content": result})
    return final

# --- demo prompts (edit these) ---
demos = [
    "List the files under ./data and count them.",
    "Read requirements.txt and summarize the dependencies in one sentence.",
    "In one sentence, what is LoRA? Do not use tools.",
]

for q in demos:
    print("\n" + "=" * 60)
    print("USER:", q)
    chat_once(q)

# Uncomment for interactive chat in Colab:
# while True:
#     q = input("\nyou> ").strip()
#     if not q or q.lower() in {"quit", "exit", "q"}:
#         break
#     chat_once(q)

print("\nDone. Edit `demos` or uncomment the input loop to keep chatting.")

Reusing model/tokenizer already in memory.

USER: List the files under ./data and count them.

--- assistant (round 1) ---
I'll list the contents of ./data for you.

invoke tool list_dir with path is ./data

--- tool: list_dir {'path': './data'} ---
example_traces.jsonl
prompt_completion.jsonl
notes.md

--- assistant (round 2) ---
There are 3 files in `./data`:
1. `example_traces.jsonl`
2. `prompt_completion.jsonl`
3. `notes.md`

USER: Read requirements.txt and summarize the dependencies in one sentence.

--- assistant (round 1) ---
I'll read requirements.txt.

invoke tool read_file with path is requirements.txt

--- tool: read_file {'path': 'requirements.txt'} ---
torch
transformers
trl
peft
bitsandbytes


--- assistant (round 2) ---
The dependencies listed are `torch`, `transformers`, `trl`, and `bitsandbytes`.

USER: In one sentence, what is LoRA? Do not use tools.

--- assistant (round 1) ---
LoRA stands for Low-Rank Adaptation, which is a technique used to increase the capacity of